# Sprint 2 Campus — Protect the FieldCare service

HelioDesk now has two callers: Dispatch and Partner. We will observe who gets through the boundary, which attempts consume an allowance, and what must never appear in shared output.

Use **one saved copy of this notebook throughout the sprint**. Work one demonstration at a time: predict, run, read the result, then explain the boundary it reveals. Each section names the matching Campus lesson. The optional project cells stay off until you are ready to test your own work; **Run all runs the supplied demos without requiring a finished assessment**.

| Campus lesson | Notebook destination | What to notice |
|---|---|---|
| API keys in plain language | §2–4 · Credentials, guard, and HTTP outcomes | A caller key and provider key serve different boundaries. |
| Adding API key auth to the scaffold | §3–4 · Guard and outcomes | Authentication precedes schema validation. |
| What is a secret and where it should live | §4 · Secrets-safety demonstration | A minimized record excludes a synthetic marker. |
| Why AI services need rate limits; Applying per-key rate limits | §5 · Apply a per-caller allowance | Callers have independent allowances pooled across routes. |
| Secured service checkpoint | §6 · Checkpoint assessment | Apply and test the policy in your own project. |

**Two folders, two purposes.** `DEMO` contains supplied, runnable instructor examples. `PROJECT` contains your editable service and any imported checkpoint. Demonstrations never edit `PROJECT`; a successful demo is not assessment completion. Source modules and the evaluation design are already in the repository. Cells make calls and inspect results; they do not manufacture Python files.

**Before running:** add `OPENROUTER_API_KEY` to Colab Secrets and enable notebook access, or use the hidden prompt. Initial Chroma indexing and supported model requests use provider quota. Generated wording and timings vary. Keep credentials out of cells and output.

Open Colab’s table of contents for section navigation. Save your personal copy in Drive for notes and outputs; use the final checkpoint cell to save your own service files.

## Campus lessons

- [API keys in plain language](https://app.notion.com/p/3ea5a55972ad81e898f1ca03d801e980?pvs=204)
- [Adding API key auth to the scaffold](https://app.notion.com/p/3ea5a55972ad81d3951cecd6d8dbd5c7?pvs=204)
- [What is a secret and where it should live](https://app.notion.com/p/3ea5a55972ad81d48e28c56f84342984?pvs=204)
- [Why AI services need rate limits](https://app.notion.com/p/3ea5a55972ad8157a8d4f24c0a8674e2?pvs=204)
- [Applying per-key rate limits](https://app.notion.com/p/3ea5a55972ad81ac920cf6a901d6388c?pvs=204)
- [Secured service checkpoint](https://app.notion.com/p/3ea5a55972ad8177b84fdac0b397fa8d?pvs=204)

## 1. Set up the cumulative project

The demo runs from supplied repository files. To carry your own routes forward, upload your Sprint 1 source ZIP, set `CHECKPOINT`, and choose a new `PROJECT` folder. Leaving it empty prepares a clearly labelled starter. The demo does not modify either project.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
BASE = Path("/content") if Path("/content").exists() else Path.cwd() / "campus-runtime"
SOURCE_REVISION = "5df838530f9220af86e36224cfdfaaadccf3beb0"
REPO = BASE / f"ai-app-dev_Mod-B-{SOURCE_REVISION[:12]}"
DEMO = BASE / f"fieldcare-campus-demo-2-{SOURCE_REVISION[:12]}"
PROJECT = BASE / "fieldcare-campus-project-2"
CHECKPOINT = ""  # Optional earlier Campus source ZIP; restore to a new PROJECT path.

# Source and instructor examples are versioned; learner PROJECT files are retained.
if not REPO.exists():
    subprocess.run(["git", "init", str(REPO)], check=True, capture_output=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", REPOSITORY, SOURCE_REVISION], check=True, capture_output=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True, capture_output=True)
actual_revision = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if actual_revision != SOURCE_REVISION:
    raise RuntimeError("Teaching source differs from the required revision. Choose a new REPO folder; existing files were preserved.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
sys.path.insert(0, str(REPO / "src"))
# An older notebook may already have imported the old package in this kernel.
loaded_module = sys.modules.get("module_b")
if loaded_module is not None and Path(loaded_module.__file__).resolve().parent != (REPO / "src/module_b").resolve():
    for module_name in list(sys.modules):
        if module_name == "module_b" or module_name.startswith("module_b."):
            del sys.modules[module_name]

from module_b.openrouter import DEFAULT_CHAT_MODEL, require_openrouter_key
from module_b.workspace import prepare_example
from module_b.campus import prepare_project, save_checkpoint
require_openrouter_key(prompt=True)
os.environ.setdefault("OPENROUTER_MODEL", DEFAULT_CHAT_MODEL)
os.environ["CAMPUS_SPRINT"] = "2"
DEMO = prepare_example(destination=DEMO, repo_root=REPO)
PROJECT = prepare_project(REPO, PROJECT, CHECKPOINT)
os.chdir(DEMO)
sys.path.insert(0, str(DEMO))
if globals().get("client") is not None:
    client.__exit__(None, None, None)
    client = None
for module_name in list(sys.modules):
    if module_name == "app" or module_name.startswith("app."):
        del sys.modules[module_name]

print("Teaching source revision:", actual_revision[:12])
print("Instructor source:", DEMO)
print("Your project:", PROJECT)
print("Project origin:", "imported checkpoint" if CHECKPOINT else "supplied starter (not a completed assessment)")

## 2. Keep caller and provider credentials separate

**Predict:** which key belongs in the caller’s `X-API-Key` header, and which is used only by FieldCare to reach OpenRouter? We generate temporary caller values without printing them. `CALLER_ENV` contains public configuration names, not credentials.

In [ ]:
import secrets
import httpx
from module_b.runtime import ServiceProcess
from app.campus_demo import CALLER_ENV
caller_keys = {label: secrets.token_urlsafe(32) for label in CALLER_ENV}
caller_env = {variable: caller_keys[label] for label, variable in CALLER_ENV.items()}
print("Configured caller labels:", list(CALLER_ENV))
print("Caller values are in memory; the provider credential remains separate.")

## 3. Attach the caller guard before the handler

Open `DEMO/app/campus_demo.py`. The repository already supplies the complete runnable example. Locate this attachment after route registration:

```python
app.add_middleware(
    SecurityMiddleware,
    caller_env=CALLER_ENV,
    protected_paths=protected_post_paths(app),
    policy=policy,
)
```

`SecurityMiddleware` runs before the handler. Start the authentication-only demo with `CAMPUS_ALLOWANCE="0"`; the public configuration value disables the demo’s limiter, not authentication. `ServiceProcess` starts one loopback Uvicorn child and checks readiness; no foreground server cell or public tunnel is needed.

In [ ]:
if globals().get("service") is not None:
    service.stop()
service = ServiceProcess(
    "app.campus_demo:create_app", factory=True, project_dir=DEMO,
    env={**caller_env, "CAMPUS_SPRINT": "2", "CAMPUS_ALLOWANCE": "0"},
    startup_timeout=120,
).start()
print("Owned local service:", service.base_url)

## 4. Compare actual HTTP outcomes

**Predict each row before running:** missing key → `401`; wrong key → `401`; recognized caller with an empty body → `422`; recognized caller with a supported question → `200` if generation succeeds. Which boundary stops each failure?

Only the supported admitted request below reaches generation. Read its actual answer and citations; `200` does not establish answer quality.

In [ ]:
payload = {"question": "Which filter and airflow checks are documented?", "equipment_id": "EQ-FC-1002"}
auth_cases = [
    ("missing caller key", {}, payload),
    ("wrong caller key", {"X-API-Key": "SYNTHETIC-NOT-A-VALID-KEY"}, payload),
    ("recognized caller, invalid body", {"X-API-Key": caller_keys["dispatch"]}, {}),
    ("recognized caller, supported question", {"X-API-Key": caller_keys["dispatch"]}, payload),
]
with httpx.Client(base_url=service.base_url, timeout=120) as client:
    for label, headers, body in auth_cases:
        result = client.post("/v1/diagnose", json=body, headers=headers)
        print(label, "→", result.status_code)
        print(result.json())
service.stop()

**Interpret:** authentication asks who may enter; validation asks whether the body follows the contract. Source inspection establishes their order. A status code is not a measurement of provider-call count.

### Secrets-safety demonstration

The next cell is an **in-memory synthetic example**, not a live request or a production log. Before running, choose which fields may remain in a shareable operational record. The marker must disappear from the minimized copy. No marker is sent to OpenRouter, written to a file, or used as a credential.

In [ ]:
import uuid
from module_b.observability import safe_record
marker = "SYNTHETIC-NOT-A-CREDENTIAL"
candidate = {
    "request_id": str(uuid.uuid4()), "route": "/v1/diagnose",
    "status_code": 401, "outcome": "rejected", "source": "none",
    "error_category": "unauthorized", "latency_ms": 4.2,
    "prompt": "A private synthetic message: " + marker,
    "headers": {"X-API-Key": marker},
}
minimized = safe_record(candidate, routes=("/v1/diagnose",))
print("Retained fields:", minimized)
print("Marker in retained record:", marker in repr(minimized))
print("Original object still contains marker:", marker in repr(candidate))

**Explain:** making a safe copy does not erase the original object or any earlier output. `.gitignore` similarly cannot erase a tracked secret from commit history. Keep real keys out of source, prompts, logs, and notebook output from the start.

### Apply the guard to your own project

Use the provided `examples/patterns/security_settings.py` as the configuration pattern: it contains `CALLER_ENV` and `POLICY`, never key values. In `PROJECT/app/main.py`, import those settings and attach the middleware after **all** your routers. Make these edits in the Files panel. The demonstration does not silently install them into your project.

```python
from module_b.security import SecurityMiddleware, protected_post_paths
from app.security_settings import CALLER_ENV, POLICY
app.add_middleware(SecurityMiddleware, caller_env=CALLER_ENV,
                   protected_paths=protected_post_paths(app), policy=POLICY)
```

Use the project request cell in §6 to verify your guard with limiting still disabled; design your own request plan first.

## 5. Apply a per-caller allowance

### Instructor demo: one caller, one allowance across routes

We will start a fresh demo process with **two admissions per caller in 60 seconds**. The window begins at that caller’s first admission. Missing/wrong keys consume nothing. Authenticated attempts consume one admission before body validation; that includes a `422`.

The short experiment uses malformed and missing-context requests to isolate the limiter from provider latency. The admitted clarification is real service behavior, not a fake model answer. **Predict:** after Dispatch uses two slots, can it avoid the limit through `/v2/diagnose`? Can Partner still enter?

In [ ]:
PRACTICE_ALLOWANCE = 2  # Change to 1–4, predict the sequence, then rerun from here.
if globals().get("service") is not None:
    service.stop()
service = ServiceProcess(
    "app.campus_demo:create_app", factory=True, project_dir=DEMO,
    env={**caller_env, "CAMPUS_SPRINT": "2", "CAMPUS_ALLOWANCE": str(PRACTICE_ALLOWANCE)},
    startup_timeout=120,
).start()

In [ ]:
clarify = {"question": "Which filter checks are documented?"}
REQUEST_PLAN = [
    ("missing key", None, "/v1/diagnose", clarify),
    ("wrong key", "wrong", "/v1/diagnose", clarify),
    ("Dispatch: malformed body", "dispatch", "/v1/diagnose", {}),
    ("Dispatch: missing context", "dispatch", "/v1/diagnose", clarify),
    ("Dispatch: other version", "dispatch", "/v2/diagnose", clarify),
    ("Partner: independent allowance", "partner", "/v1/diagnose", clarify),
]
practice_responses = []
with httpx.Client(base_url=service.base_url, timeout=30) as client:
    for label, caller, path, body in REQUEST_PLAN:
        headers = {} if caller is None else {"X-API-Key": caller_keys.get(caller, "SYNTHETIC-WRONG")}
        result = client.post(path, json=body, headers=headers)
        practice_responses.append(result)
        print(label, path, "→", result.status_code, result.json().get("status"),
              "Retry-After:", result.headers.get("retry-after"))

**Read the result:** at the default allowance, expect `401, 401, 422, 200, 429, 200`. Dispatch’s second admission is a clarification, and its version change shares the exhausted allowance. Partner’s bucket is independent. Changing the allowance can change that sequence—explain the change using counted admissions.

### Optional: observe renewal without restarting

Set `OBSERVE_RENEWAL=True` to wait through the returned `Retry-After` and retry Dispatch in this **same process**. This can take about 60 seconds. Restarting would create a new bucket and would not demonstrate elapsed-window renewal.

In [ ]:
OBSERVE_RENEWAL = False
if OBSERVE_RENEWAL:
    import time
    limited = next((r for r in practice_responses if r.status_code == 429), None)
    if limited is None:
        print("No 429 in this plan: design a burst that exhausts the selected allowance first.")
    else:
        wait_seconds = int(limited.headers["retry-after"])
        print("Waiting", wait_seconds, "seconds in the same service process…", flush=True)
        time.sleep(wait_seconds)
        result = httpx.post(service.base_url + "/v1/diagnose", json=clarify,
                            headers={"X-API-Key": caller_keys["dispatch"]}, timeout=30)
        print("After window elapsed:", result.status_code, result.json().get("status"))
else:
    print("Renewal was not observed in this run; Retry-After alone is not a renewal test.")
service.stop()

### Your independent practice

In `PROJECT/app/security_settings.py`, use `LimitPolicy` to choose an allowance from one to four with a 60-second window. Preserve your authenticated routes and design your own ordered burst. Use the request runner below after reading the independent-practice brief. Include a real supported request when checking your service end to end; the instructor’s clarification-only burst does not establish generation behavior.

## 6. Checkpoint assessment: secure a weekend partner pilot

In your own project, apply the two-request, 60-second policy in the assessment brief. Preserve previous routes. First write the caller, path, body, predicted result, and reason for each step. Demonstrate rejection, schema validation, real admitted generation, exhaustion, and another caller’s independent allowance.

`MY_REQUEST_PLAN` is deliberately empty: deciding the sequence is your task. Each row uses `label`, `caller` (`dispatch`, `partner`, `wrong`, or `None`), `path`, and `body`. Set `RUN_MY_PROJECT=True` only after editing your source and plan. The same runner can check your authentication-only and independent-practice stages before assessment.

In [ ]:
RUN_MY_PROJECT = False
MY_REQUEST_PLAN = []  # Supply your ordered rows and keep predictions in a text cell.
if RUN_MY_PROJECT:
    if not MY_REQUEST_PLAN:
        raise ValueError("Write your request plan before running your project.")
    with ServiceProcess(project_dir=PROJECT, env=caller_env, startup_timeout=120) as project_service:
        with httpx.Client(base_url=project_service.base_url, timeout=120) as project_client:
            for step in MY_REQUEST_PLAN:
                caller = step["caller"]
                headers = {} if caller is None else {"X-API-Key": caller_keys.get(caller, "SYNTHETIC-WRONG")}
                result = project_client.post(step["path"], json=step["body"], headers=headers)
                print(step["label"], result.status_code, result.json(),
                      "Retry-After:", result.headers.get("retry-after"))
else:
    print("Project verification has not run. The instructor demo does not complete the assessment.")

## 7. Save and carry the secured service forward

Keep your policy, protected route inventory, ordered predictions and observations, and one limitation of the in-memory limiter. Record the actual provider-backed response separately from the limiter-only demonstration. Save your own project using the final export below.

## Save your Campus checkpoint

Save the notebook in Drive to retain predictions, outputs, and notes. The export below packages **your `PROJECT` source and data**, not the instructor demo. If you only ran the demos, it contains an unchanged starter. Review your editable files before sharing; an export allowlist cannot detect a credential pasted into Python code.

Download the printed ZIP using Colab’s Files panel. In the next sprint, set `CHECKPOINT` to its uploaded path and use a new `PROJECT` folder. For a same-runtime rerun after restoring, clear `CHECKPOINT` to reuse the existing project. Never overwrite an existing workspace to recover from an error.

In [ ]:
if globals().get("service") is not None:
    service.stop()
CHECKPOINT_OUT = save_checkpoint(PROJECT, BASE / "exports", sprint=2)
print("Your project checkpoint:", CHECKPOINT_OUT)
print("Save the notebook separately for observations and any safe log records you displayed.")

## Watch out for

- **Missing package after setup:** reopen the original Campus link for the updated setup. It loads a versioned source folder and refreshes an older imported package without overwriting your project. Run setup, then rerun the demonstration sections in order. Preserve your saved notebook observations.
- **Provider error or quota limit:** keep the actual error, check your provider configuration, and rerun only the affected call. Do not substitute a sample answer.
- **Changing Python files has no effect:** stop the owned process and restart it. The supplied demo and your `app.main` are different applications; check which one the request cell starts.
- **Checkpoint destination already exists:** clear `CHECKPOINT` to reuse it, or choose a new `PROJECT` path to restore. Your existing files are preserved.

## Summary

Use the observations above to explain this sprint’s service boundaries. Complete your own project tasks using the lesson brief, then keep its source and run evidence together for the next sprint.